# 04 - Carga Ouro -> Postgres

Lê as tabelas Iceberg da camada **Ouro** (dimensões e fato de vendas) e
grava no banco analítico Postgres (`metastore` ou um database dedicado,
ex: `analytics`), via JDBC, para consumo por ferramentas de BI
(Metabase/Grafana — futuro).

Tabelas de destino no Postgres (schema `ouro`):
- `ouro.tab_dim_produto`
- `ouro.tab_dim_categoria`
- `ouro.tab_dim_data`
- `ouro.tab_fato_vendas`

In [ ]:
from pyspark.sql import SparkSession

## 1. Configuração da SparkSession

Mantemos o catálogo Iceberg `lakehouse` para leitura da Ouro, e usamos
o conector JDBC do Spark (`df.write.jdbc`) para escrever no Postgres.

In [ ]:
spark = (
    SparkSession.builder
    .appName("ouro-to-postgres")
    .config("spark.jars.packages",
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1,"
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
            "org.postgresql:postgresql:42.7.4")

    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "jdbc")
    .config("spark.sql.catalog.lakehouse.uri", "jdbc:postgresql://postgres:5432/metastore")
    .config("spark.sql.catalog.lakehouse.jdbc.user", "dba_admin")
    .config("spark.sql.catalog.lakehouse.jdbc.password", "Agent9-Backtalk6-Zestfully5-Luxury1-Calculus2")
    .config("spark.sql.catalog.lakehouse.jdbc.driver", "org.postgresql.Driver")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3a://lakehouse/warehouse")

    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin_minio")
    .config("spark.hadoop.fs.s3a.secret.key", "Grunt9-Relenting6-Hula5-Catcall9-Residue3")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")

    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

## 2. Parâmetros de conexão JDBC com o Postgres (destino analítico)

> Recomenda-se criar um database dedicado (`analytics`) separado do
> `metastore` (que é exclusivo do catálogo Iceberg), para não misturar
> metadados internos do Iceberg com dados de negócio.
>
> Crie o database antes de rodar este notebook:
> ```sql
> CREATE DATABASE analytics;
> ```

In [ ]:
PG_URL = "jdbc:postgresql://postgres:5432/analytics"
PG_PROPERTIES = {
    "user": "dba_admin",
    "password": "Agent9-Backtalk6-Zestfully5-Luxury1-Calculus2",
    "driver": "org.postgresql.Driver",
}
PG_SCHEMA = "ouro"

## 3. Leitura das tabelas Ouro (Iceberg)

In [ ]:
df_dim_produto   = spark.table("lakehouse.ouro.tab_dim_produto")
df_dim_categoria = spark.table("lakehouse.ouro.tab_dim_categoria")
df_dim_data      = spark.table("lakehouse.ouro.tab_dim_data")
df_fato_vendas   = spark.table("lakehouse.ouro.tab_fato_vendas")

for nome, df in [
    ("tab_dim_produto", df_dim_produto),
    ("tab_dim_categoria", df_dim_categoria),
    ("tab_dim_data", df_dim_data),
    ("tab_fato_vendas", df_fato_vendas),
]:
    print(f"{nome}: {df.count()} registros")

## 4. Criação do schema `ouro` no Postgres

O `df.write.jdbc` não cria schemas automaticamente — apenas tabelas
dentro de um schema já existente. Criamos o schema via uma conexão
direta com `psycopg2` (alternativa: rodar `CREATE SCHEMA` manualmente
no `psql`/DBeaver antes da primeira execução).

In [ ]:
import psycopg2

conn = psycopg2.connect(
    host="postgres",
    port=5432,
    dbname="analytics",
    user=PG_PROPERTIES["user"],
    password=PG_PROPERTIES["password"],
)
conn.autocommit = True

with conn.cursor() as cur:
    cur.execute(f"CREATE SCHEMA IF NOT EXISTS {PG_SCHEMA}")

conn.close()
print(f"Schema '{PG_SCHEMA}' OK no database 'analytics'")

## 5. Carga das dimensões (overwrite)

Dimensões são pequenas e totalmente recalculadas a cada execução do
pipeline Ouro — `overwrite` mantém o Postgres sempre sincronizado com
o estado mais recente do Iceberg.

In [ ]:
def carregar_tabela(df, nome_tabela, modo="overwrite"):
    (
        df.write
        .mode(modo)
        .jdbc(
            url=PG_URL,
            table=f"{PG_SCHEMA}.{nome_tabela}",
            properties=PG_PROPERTIES,
        )
    )
    print(f"Tabela '{PG_SCHEMA}.{nome_tabela}' carregada ({modo}).")


carregar_tabela(df_dim_produto, "tab_dim_produto")
carregar_tabela(df_dim_categoria, "tab_dim_categoria")
carregar_tabela(df_dim_data, "tab_dim_data")

## 6. Carga do fato (overwrite)

Assim como as dimensões, o fato é recalculado a partir da Prata a cada
execução (ver notebook 03), então `overwrite` é consistente aqui.
Para um cenário de fato realmente incremental, troque para `append`
com deduplicação por `sk_pedido` no lado Postgres (ex: `ON CONFLICT`).

In [ ]:
carregar_tabela(df_fato_vendas, "tab_fato_vendas")

## 7. Verificação no Postgres

In [ ]:
df_check = spark.read.jdbc(
    url=PG_URL,
    table=f"{PG_SCHEMA}.tab_fato_vendas",
    properties=PG_PROPERTIES,
)
df_check.show(10)
print(f"Total no Postgres ({PG_SCHEMA}.tab_fato_vendas): {df_check.count()}")

In [ ]:
df_resumo = spark.read.jdbc(
    url=PG_URL,
    table=f"(SELECT categoria, COUNT(*) AS qtd FROM {PG_SCHEMA}.tab_dim_produto GROUP BY categoria) AS resumo",
    properties=PG_PROPERTIES,
)
df_resumo.show()

In [ ]:
spark.stop()